<a href="https://colab.research.google.com/github/alizaabbasi/Project-Market-basket-analysis-/blob/main/Market_Basket_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [ ]:
from google.colab import ai
response = ai.generate_text("What is the capital of France?")

In [ ]:
seconds_in_a_day = 24 * 60 * 60
seconds_in_a_day

86400

In [ ]:
seconds_in_a_week = 7 * seconds_in_a_day
seconds_in_a_week

604800

In [3]:
import os
from google.colab import userdata

# 1. Install Spark and the Kaggle downloader
!pip install pyspark kaggle

# 2. Pull the token from your blue-toggled secret box
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

# 3. Create a folder for Kaggle and move the token there (Kaggle requires this specific folder setup)
!mkdir -p ~/.kaggle
!echo '{"token": "'"$KAGGLE_API_TOKEN"'"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

print("Spark installed and Kaggle token loaded successfully!")

Spark installed and Kaggle token loaded successfully!


In [19]:
import os

# Re-introducing dummy strings to protect sensitive information for final submission
os.environ['KAGGLE_USERNAME'] = "xxxxxx"
os.environ['KAGGLE_KEY'] = "xxxxxx"

print("Credentials safely sanitized for GitHub deployment!")

Credentials safely sanitized for GitHub deployment!


In [4]:
# Download the specific IMDB dataset required for Project 2
!kaggle datasets download -d asaniczka/tmdb-movies-dataset-2023-930k-movies
!unzip -o tmdb-movies-dataset-2023-930k-movies.zip -d imdb_data

print("Dataset downloaded and unzipped successfully!")

Dataset URL: https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies
License(s): ODC Attribution License (ODC-By)
100% 246M/246M [00:02<00:00, 99.1MB/s]

Archive:  tmdb-movies-dataset-2023-930k-movies.zip
  inflating: imdb_data/TMDB_movie_dataset_v11.csv  
Dataset downloaded and unzipped successfully!


In [5]:
import os
from google.colab import userdata

# 1. Securely fetch your token
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

# 2. Setup the Kaggle folder exactly how the system expects it
!mkdir -p ~/.kaggle
!echo '{"token": "'"$KAGGLE_API_TOKEN"'"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# 3. Download and unzip the IMDB dataset
!kaggle datasets download -d asaniczka/tmdb-movies-dataset-2023-930k-movies
!unzip -o tmdb-movies-dataset-2023-930k-movies.zip -d imdb_data

print("Dataset downloaded successfully!")

Dataset URL: https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies
License(s): ODC Attribution License (ODC-By)
tmdb-movies-dataset-2023-930k-movies.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  tmdb-movies-dataset-2023-930k-movies.zip
  inflating: imdb_data/TMDB_movie_dataset_v11.csv  
Dataset downloaded successfully!


In [6]:
from pyspark.sql import SparkSession

# 1. Define the Global Variable requested by the Professor
USE_SUBSAMPLE = True
SUBSAMPLE_FRACTION = 0.01 # We will use 1% of the data just to build and test our code

# 2. Start our Spark distributed computing session
spark = SparkSession.builder \
    .appName("IMDB_Market_Basket_Analysis") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# 3. Load the unzipped CSV file into Spark
file_path = "imdb_data/*.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)

# 4. Apply the subsampling logic
if USE_SUBSAMPLE:
    print(f"Subsampling triggered: Using {SUBSAMPLE_FRACTION * 100}% of the data for testing.")
    df = df.sample(withReplacement=False, fraction=SUBSAMPLE_FRACTION, seed=42)
else:
    print("Using the full dataset for final execution.")

# 5. Show the first 5 rows to confirm it worked!
df.show(5)

Subsampling triggered: Using 1.0% of the data for testing.
+------+--------------------+------------+----------+--------+------------+---------+-------+-----+--------------------+---------+--------------------+---------+-----------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|    id|               title|vote_average|vote_count|  status|release_date|  revenue|runtime|adult|       backdrop_path|   budget|            homepage|  imdb_id|original_language|      original_title|            overview|popularity|         poster_path|             tagline|              genres|production_companies|production_countries|    spoken_languages|            keywords|
+------+--------------------+------------+----------+--------+------------+---------+-------+-----+--------------------+---------+--------------------+---------+-----------------+--

In [11]:
import os
from google.colab import userdata

# 1. Make sure we are still authenticated with Kaggle
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

# 2. Download the CORRECT IMDB dataset that contains the Star columns
!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
!unzip -o imdb-dataset-of-top-1000-movies-and-tv-shows.zip -d real_imdb_data

# 3. Load the newly downloaded, correct file into Spark
file_path = "real_imdb_data/imdb_top_1000.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)

print("Correct dataset loaded! Here are the actual columns inside it:")
print(df.columns)

Dataset URL: https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
License(s): CC0-1.0
100% 175k/175k [00:00<00:00, 53.8MB/s]

Archive:  imdb-dataset-of-top-1000-movies-and-tv-shows.zip
  inflating: real_imdb_data/imdb_top_1000.csv  
Correct dataset loaded! Here are the actual columns inside it:
['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate', 'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director', 'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross']


In [12]:
# 1. Select only the four actor columns specified in the project
stars_df = df.select("Star1", "Star2", "Star3", "Star4")

# 2. Convert the Spark DataFrame into an RDD (Resilient Distributed Dataset)
# RDDs are necessary for the low-level MapReduce-style operations we are going to do
raw_rdd = stars_df.rdd

# 3. Create a function to clean up the rows
def format_basket(row):
    # This takes the row, extracts the actor names, and ignores any empty (None) values
    basket = [actor for actor in row if actor is not None]
    return basket

# 4. Apply the function to our RDD
baskets_rdd = raw_rdd.map(format_basket)

# 5. Filter out any baskets that have less than 2 actors
# (Because we are looking for frequent pairs, a movie with only 1 actor is useless to us)
baskets_rdd = baskets_rdd.filter(lambda basket: len(basket) >= 2)

# Show the first 5 formatted baskets to confirm it worked!
print("Here are your first 5 baskets:")
for basket in baskets_rdd.take(5):
    print(basket)

Here are your first 5 baskets:
['Tim Robbins', 'Morgan Freeman', 'Bob Gunton', 'William Sadler']
['Marlon Brando', 'Al Pacino', 'James Caan', 'Diane Keaton']
['Christian Bale', 'Heath Ledger', 'Aaron Eckhart', 'Michael Caine']
['Al Pacino', 'Robert De Niro', 'Robert Duvall', 'Diane Keaton']
['Henry Fonda', 'Lee J. Cobb', 'Martin Balsam', 'John Fiedler']


In [13]:
import itertools

# We define a global support threshold for our dataset
GLOBAL_SUPPORT = 50

def local_apriori(iterator):
    # --- Step 1: Store the Baskets ---
    baskets_list = list(iterator)

    # If this specific partition of data is empty, stop immediately
    if not baskets_list:
        return []

    # SON Algorithm Rule: Scale down the support threshold for this small chunk
    # e.g., if this chunk has 10% of the total data, it needs 10% of the global support
    # We estimate local support dynamically based on the chunk size relative to a typical partition
    local_support = max(1, int(GLOBAL_SUPPORT * 0.01))

    # --- Step 2: Pass 1 - Count Single Actors ---
    actor_counts = {}
    for basket in baskets_list:
        for actor in basket:
            if actor in actor_counts:
                actor_counts[actor] += 1
            else:
                actor_counts[actor] = 1

    # --- Step 3: Filter Frequent Single Actors ---
    frequent_singletons = set()
    for actor, count in actor_counts.items():
        if count >= local_support:
            frequent_singletons.add(actor)

    # --- Step 4 & 5: Pass 2 - Generate and Count Candidate Pairs ---
    pair_counts = {}
    for basket in baskets_list:
        # Filter to keep only individual actors who passed Pass 1
        frequent_actors_in_basket = [actor for actor in basket if actor in frequent_singletons]
        frequent_actors_in_basket = sorted(frequent_actors_in_basket)

        # Generate pairs out of the remaining frequent actors
        pairs_in_basket = itertools.combinations(frequent_actors_in_basket, 2)

        for pair in pairs_in_basket:
            if pair in pair_counts:
                pair_counts[pair] += 1
            else:
                pair_counts[pair] = 1

    # --- Step 6: Filter and Return Frequent Pairs ---
    frequent_pairs = []
    for pair, count in pair_counts.items():
        if count >= local_support:
            frequent_pairs.append(pair)

    return frequent_pairs

print("The local_apriori function has been successfully defined in your notebook!")

The local_apriori function has been successfully defined in your notebook!


In [17]:
import time

print("Starting Pass 1: Finding Candidate Pairs...")
start_time = time.time()

# --- PASS 1: MapPartitions ---
# Spark sends your local_apriori function to every chunk of data simultaneously
# Then .distinct() removes duplicates, and .collect() brings the candidates back to you
candidate_pairs = baskets_rdd.mapPartitions(local_apriori).distinct().collect()

# We convert it to a set for incredibly fast lookups in Pass 2
candidate_set = set(candidate_pairs)
print(f"Pass 1 Complete! Found {len(candidate_set)} candidate pairs.")

# --- PASS 2: Global Counting ---
print("Starting Pass 2: Counting Candidates Globally...")

# We "broadcast" the candidates to all Spark worker nodes so they all have a copy
broadcast_candidates = spark.sparkContext.broadcast(candidate_set)

def count_global_occurrences(basket):
    # This function looks at one movie, generates pairs, and checks if they are in our candidates
    found_candidates = []
    basket_sorted = sorted(basket)

    # Check all combinations in this movie
    for pair in itertools.combinations(basket_sorted, 2):
        if pair in broadcast_candidates.value:
            # If we found a candidate, emit (pair, 1) to count it
            found_candidates.append((pair, 1))

    return found_candidates

# flatMap runs our counting function, reduceByKey adds up all the 1s across the whole cluster!
global_counts = baskets_rdd.flatMap(count_global_occurrences).reduceByKey(lambda a, b: a + b)

# Finally, we filter out any pairs that didn't meet the GLOBAL_SUPPORT threshold
final_frequent_pairs = global_counts.filter(lambda pair_count: pair_count[1] >= GLOBAL_SUPPORT).collect()

end_time = time.time()
execution_time = end_time - start_time

print(f"Pass 2 Complete! Algorithm finished in {execution_time:.2f} seconds.")
print(f"Success! Found {len(final_frequent_pairs)} truly frequent collaborating actor pairs.")

# Let's see the top 10 most frequent actor pairs!
final_frequent_pairs.sort(key=lambda x: x[1], reverse=True)
print("\n--- Top 10 Frequent Actor Pairs ---")
for pair, count in final_frequent_pairs[:10]:
    print(f"{pair}: {count} movies")

Starting Pass 1: Finding Candidate Pairs...
Pass 1 Complete! Found 5835 candidate pairs.
Starting Pass 2: Counting Candidates Globally...
Pass 2 Complete! Algorithm finished in 1.03 seconds.
Success! Found 25 truly frequent collaborating actor pairs.

--- Top 10 Frequent Actor Pairs ---
('Daniel Radcliffe', 'Rupert Grint'): 6 movies
('Daniel Radcliffe', 'Emma Watson'): 5 movies
('Emma Watson', 'Rupert Grint'): 5 movies
('Joe Pesci', 'Robert De Niro'): 4 movies
('Tim Allen', 'Tom Hanks'): 4 movies
('Al Pacino', 'Diane Keaton'): 3 movies
('Christian Bale', 'Michael Caine'): 3 movies
('Al Pacino', 'Robert De Niro'): 3 movies
('Elijah Wood', 'Ian McKellen'): 3 movies
('Elijah Wood', 'Orlando Bloom'): 3 movies


In [15]:
# 1. Reload the data without any subsampling (USE_SUBSAMPLE = False)
file_path = "real_imdb_data/imdb_top_1000.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)

# 2. Recreate the baskets_rdd using the full dataset
stars_df = df.select("Star1", "Star2", "Star3", "Star4")
raw_rdd = stars_df.rdd

def format_basket(row):
    basket = [actor for actor in row if actor is not None]
    return basket

baskets_rdd = raw_rdd.map(format_basket).filter(lambda basket: len(basket) >= 2)

print("Step 1 Complete: Full dataset loaded into baskets_rdd!")

Step 1 Complete: Full dataset loaded into baskets_rdd!


In [16]:
import itertools

# --- WE CHANGED THIS FROM 50 TO 3 ---
GLOBAL_SUPPORT = 3

def local_apriori(iterator):
    baskets_list = list(iterator)
    if not baskets_list:
        return []

    local_support = max(1, int(GLOBAL_SUPPORT * 0.01))

    # Pass 1
    actor_counts = {}
    for basket in baskets_list:
        for actor in basket:
            if actor in actor_counts:
                actor_counts[actor] += 1
            else:
                actor_counts[actor] = 1

    frequent_singletons = set()
    for actor, count in actor_counts.items():
        if count >= local_support:
            frequent_singletons.add(actor)

    # Pass 2
    pair_counts = {}
    for basket in baskets_list:
        frequent_actors_in_basket = [actor for actor in basket if actor in frequent_singletons]
        frequent_actors_in_basket = sorted(frequent_actors_in_basket)
        pairs_in_basket = itertools.combinations(frequent_actors_in_basket, 2)

        for pair in pairs_in_basket:
            if pair in pair_counts:
                pair_counts[pair] += 1
            else:
                pair_counts[pair] = 1

    frequent_pairs = []
    for pair, count in pair_counts.items():
        if count >= local_support:
            frequent_pairs.append(pair)

    return frequent_pairs

print(f"Step 2 Complete: local_apriori function updated with GLOBAL_SUPPORT = {GLOBAL_SUPPORT}")

Step 2 Complete: local_apriori function updated with GLOBAL_SUPPORT = 3


In [18]:
print("Running the algorithm with new settings...")

# --- PASS 1 ---
candidate_pairs = baskets_rdd.mapPartitions(local_apriori).distinct().collect()
candidate_set = set(candidate_pairs)
print(f"Pass 1 Complete! Found {len(candidate_set)} candidate pairs.")

# --- PASS 2 ---
broadcast_candidates = spark.sparkContext.broadcast(candidate_set)

# (We re-evaluate global counts with the full dataset)
global_counts = baskets_rdd.flatMap(count_global_occurrences).reduceByKey(lambda a, b: a + b)
final_frequent_pairs = global_counts.filter(lambda pair_count: pair_count[1] >= GLOBAL_SUPPORT).collect()

# Sort and print
final_frequent_pairs.sort(key=lambda x: x[1], reverse=True)
print("\n--- Top 10 Frequent Actor Pairs ---")
for pair, count in final_frequent_pairs[:10]:
    print(f"{pair}: {count} movies")

Running the algorithm with new settings...
Pass 1 Complete! Found 5835 candidate pairs.

--- Top 10 Frequent Actor Pairs ---
('Daniel Radcliffe', 'Rupert Grint'): 6 movies
('Daniel Radcliffe', 'Emma Watson'): 5 movies
('Emma Watson', 'Rupert Grint'): 5 movies
('Joe Pesci', 'Robert De Niro'): 4 movies
('Tim Allen', 'Tom Hanks'): 4 movies
('Al Pacino', 'Diane Keaton'): 3 movies
('Christian Bale', 'Michael Caine'): 3 movies
('Al Pacino', 'Robert De Niro'): 3 movies
('Elijah Wood', 'Ian McKellen'): 3 movies
('Elijah Wood', 'Orlando Bloom'): 3 movies
